# E-commerce sales analysis

This notebook combines two years of Online Retail transactions, checks and cleans the records, measures sales performance, builds RFM customer segments, and exports tables for reporting. Each row in the source data represents an item line within an invoice; a single invoice can contain multiple rows.


In [4]:
import pandas as pd

## Load the two annual source sheets

The Online Retail workbook contains transaction-level data: each row is one product line in an invoice, with the invoice number, product code and description, quantity, date, unit price, customer ID, and country. The next cells read the 2009–2010 sheet and then the 2010–2011 sheet into separate Pandas DataFrames before they are combined.


In [5]:
file_path = "online_retail_II.xlsx"

df_2009_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

In [6]:
df_2009_2010.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


The preceding cell previews the 2009–2010 sheet. The next two cells load and preview the 2010–2011 sheet, confirming that both annual imports have the expected structure before they are combined.


In [7]:
df_2010_2011 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

In [8]:
df_2010_2011.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Combine and inspect the source data

The two annual DataFrames are appended row by row. `ignore_index=True` creates one continuous row index. The following checks display the table size, its first and last records, verify that no rows were lost during the append, and inspect column names, types, summary statistics, and example records.


In [9]:
df = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

In [10]:
df.shape

(1067371, 8)

In [11]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [12]:
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


In [13]:
len(df_2009_2010) + len(df_2010_2011) == len(df)

True

In [14]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [16]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [17]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Assess data quality

These checks count missing values and calculate the percentage of missing values in every column. They show which fields need attention before analysis.


In [18]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [19]:
(df.isnull().mean() * 100).round(2)

Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.77
Country         0.00
dtype: float64

### Diagnose duplicate transaction rows

This section identifies exact duplicate rows, reviews sample duplicates, and groups them by invoice and product code. The grouped count distinguishes repeated line items from likely duplicate records and helps evaluate their possible effect on revenue.


In [20]:
df.duplicated().sum()

np.int64(34335)

In [21]:
df[df.duplicated()].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom


In [22]:
duplicates = df[df.duplicated(keep=False)]

In [23]:
duplicates.sort_values(
    ["Invoice", "StockCode", "InvoiceDate"]
).head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [24]:
duplicates.groupby(
    ["Invoice", "StockCode"]
).size().sort_values(ascending=False).head(20)

Invoice  StockCode
555524   22698        20
         22697        12
536412   21448        10
537646   22585        10
537224   70007        10
536464   22866         8
537154   22727         8
537202   22147         8
502660   17021         8
572861   22775         8
536874   22866         6
537051   22727         6
538514   21756         6
536796   21967         6
         22376         6
525065   20894         6
537421   22585         6
537422   22585         6
537144   22553         6
533841   22906         6
dtype: int64

### Measure the effect of duplicates

Revenue is calculated per transaction line as quantity × unit price. The code then compares total revenue before and after removing exact duplicate rows, reporting both the currency difference and its percentage of the original total. A deduplicated working copy, `df_clean`, is created for the remaining analysis.


In [25]:
df["Revenue"] = df["Quantity"] * df["Price"]

In [26]:
total_revenue_with_duplicates = df["Revenue"].sum()
total_revenue_with_duplicates

np.float64(19287250.568)

In [27]:
df_no_duplicates = df.drop_duplicates()

total_revenue_without_duplicates = df_no_duplicates["Revenue"].sum()

total_revenue_without_duplicates

np.float64(18855533.698)

In [28]:
difference = (
    total_revenue_with_duplicates
    - total_revenue_without_duplicates
)

difference

np.float64(431716.87000000104)

In [29]:
percentage_difference = (
    difference / total_revenue_with_duplicates
) * 100

percentage_difference

np.float64(2.2383536133256556)

In [30]:
df_clean = df.drop_duplicates().copy()

In [31]:
df_clean.shape

(1033036, 9)

In [32]:
df_clean.duplicated().sum()

np.int64(0)

## Clean and profile key fields

The following cells examine transactions without a customer identifier and show where they occur. These rows remain in `df_clean` for sales analysis, but cannot be assigned to an individual customer in the later RFM analysis.


In [33]:
df_clean[df_clean["Customer ID"].isna()].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,-0.00
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,-0.00
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,-0.00
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,-0.00
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,0.55
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom,0.85
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,2.95
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,15.00
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,8.85
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,1.65


In [34]:
df_clean[df_clean["Customer ID"].isna()]["Country"].value_counts().head(10)

Country
United Kingdom          232206
EIRE                      1653
Hong Kong                  360
Unspecified                231
France                     128
Switzerland                117
United Arab Emirates       114
Portugal                   114
Bahrain                     67
Israel                      47
Name: count, dtype: int64

### Remove invalid or free-price rows

The code counts and inspects zero or negative prices, checks which product descriptions and invoice types are involved, then keeps only rows with a positive price. This prevents zero- or negative-priced records from distorting revenue measures. Negative quantities are intentionally retained because they can represent returns or cancellations.


In [35]:
(df_clean["Price"] <= 0).sum()

np.int64(6019)

In [36]:
df_clean[df_clean["Price"] <= 0].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,-0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,-0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,-0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,-0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,-0.0
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,0.0
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,-0.0
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,-0.0
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,0.0
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,-0.0


In [37]:
(df_clean["Price"] == 0).sum()

np.int64(6014)

In [38]:
df_clean.loc[
    df_clean["Price"] == 0,
    "Description"
].value_counts(dropna=False).head(30)

Description
NaN                                4275
check                               160
?                                    90
damages                              83
damaged                              81
found                                28
missing                              27
sold as set on dotcom                20
Damaged                              17
adjustment                           16
OWL DOORSTOP                         14
dotcom                               12
POLYESTER FILLER PAD 45x45cm         12
POLYESTER FILLER PAD 40x40cm         10
Found                                 9
FRENCH BLUE METAL DOOR SIGN 1         9
smashed                               9
Unsaleable, destroyed.                9
thrown away                           9
checked                               8
RED KITCHEN SCALES                    8
RECIPE BOX PANTRY YELLOW DESIGN       8
IVORY KITCHEN SCALES                  8
amazon                                8
PICNIC BASKET WICKER LARGE  

In [39]:
df_clean.loc[
    df_clean["Price"] == 0,
    "Invoice"
].astype(str).str.startswith("C").value_counts()

Invoice
False    6014
Name: count, dtype: int64

In [40]:
df_clean = df_clean[df_clean["Price"] > 0].copy()

In [41]:
(df_clean["Price"] <= 0).sum()

np.int64(0)

In [42]:
df_clean.shape

(1027017, 9)

### Identify cancellation transactions

Negative quantities are inspected and compared with invoice numbers beginning with `C`. A Boolean `IsCancellation` flag is then created from that invoice-number convention. The counts, cross-tabulation, and final filter check whether cancellation status aligns with negative quantities and whether any unusual cases exist.


In [43]:
(df_clean["Quantity"] < 0).sum()

np.int64(19103)

In [44]:
df_clean[df_clean["Quantity"] < 0].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.40
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,-9.90
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,-17.00
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,-12.60
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.40
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,-15.00
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,-15.00
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia,-20.40
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,-35.40
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom,-12.75


In [45]:
df_clean.loc[
    df_clean["Quantity"] < 0,
    "Invoice"
].astype(str).str.startswith("C").value_counts()

Invoice
True    19103
Name: count, dtype: int64

In [46]:
df_clean["IsCancellation"] = (
    df_clean["Invoice"]
    .astype(str)
    .str.startswith("C")
)

In [47]:
df_clean["IsCancellation"].value_counts()

IsCancellation
False    1007913
True       19104
Name: count, dtype: int64

In [48]:
pd.crosstab(
    df_clean["IsCancellation"],
    df_clean["Quantity"] < 0
)

Quantity,False,True
IsCancellation,,
False,1007913,0
True,1,19103


In [49]:
df_clean[
    (df_clean["IsCancellation"]) &
    (df_clean["Quantity"] >= 0)
]


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,IsCancellation
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom,373.57,True


### Check missing product descriptions

These diagnostics list transactions with a missing product description, identify their product codes, and count them. No imputation is applied: the records are preserved for overall sales calculations, while product summaries that group by description will naturally exclude missing descriptions.


In [50]:
df_clean[df_clean["Description"].isna()].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,IsCancellation


In [51]:
df_clean[df_clean["Description"].isna()]["StockCode"].value_counts().head(20)

Series([], Name: count, dtype: int64)

In [52]:
df_clean["Description"].isna().sum()

np.int64(0)

### Summarize customer and product coverage

The code counts unique customers and products, quantifies transactions without a customer ID, and totals the revenue from identified versus anonymous customers. It then creates a customer-only subset for customer-level analysis. This initial subset is later rebuilt after the date features are added.


In [53]:
df_clean["Customer ID"].nunique()

5939

In [54]:
df_clean["StockCode"].nunique()

4932

In [55]:
df_clean["Customer ID"].isna().sum()

np.int64(229202)

In [56]:
df_clean["Customer ID"].isna().mean() * 100

np.float64(22.31725472898696)

In [57]:
df_clean.groupby(
    df_clean["Customer ID"].isna()
)["Revenue"].sum()

Customer ID
False    1.628999e+07
True     2.724219e+06
Name: Revenue, dtype: float64

### Create an initial customer-only subset

This cell keeps only rows with a customer ID. It creates an initial customer-linked subset, which is rebuilt later after the date features have been added for RFM analysis.


In [58]:
df_customers = df_clean[df_clean["Customer ID"].notna()].copy()

## Create time features

`InvoiceDate` is converted to a true datetime value. Year, numeric month, month name, day of month, weekday name, and hour are extracted so sales can be grouped by different time periods. The final preview confirms the new columns.


In [59]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

In [60]:
df_clean["Year"] = df_clean["InvoiceDate"].dt.year

In [61]:
df_clean["Month"] = df_clean["InvoiceDate"].dt.month

In [62]:
df_clean["Month_Name"] = df_clean["InvoiceDate"].dt.month_name()

In [63]:
df_clean["Day"] = df_clean["InvoiceDate"].dt.day

In [64]:
df_clean["Day_Name"] = df_clean["InvoiceDate"].dt.day_name()

In [65]:
df_clean["Hour"] = df_clean["InvoiceDate"].dt.hour

In [66]:
df_clean.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,IsCancellation,Year,Month,Month_Name,Day,Day_Name,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,False,2009,12,December,1,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,False,2009,12,December,1,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,False,2009,12,December,1,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,False,2009,12,December,1,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,False,2009,12,December,1,Tuesday,7


### Confirm the analysis period

The earliest and latest transaction timestamps define the period covered by the cleaned dataset. These dates also provide the reference point for customer recency later in the notebook.


In [67]:
df_clean["InvoiceDate"].min()

Timestamp('2009-12-01 07:45:00')

In [68]:
df_clean["InvoiceDate"].max()

Timestamp('2011-12-09 12:50:00')

## Calculate headline sales metrics

Net revenue includes all retained transaction lines, so cancellation lines reduce it. Gross revenue excludes invoice numbers flagged as cancellations. `returned_amount` is therefore a negative adjustment rather than an absolute return value. Order count is the number of distinct invoice identifiers (including cancellation invoices), and average order value is net revenue divided by that count. The dictionary gathers these measures in one result.


In [69]:
total_revenue = df_clean["Revenue"].sum()

total_revenue

np.float64(19014209.838)

In [70]:
gross_revenue = df_clean.loc[
    ~df_clean["IsCancellation"],
    "Revenue"
].sum()

gross_revenue

np.float64(20476260.448)

In [71]:
returned_amount = df_clean.loc[
    df_clean["IsCancellation"],
    "Revenue"
].sum()

returned_amount

np.float64(-1462050.61)

In [72]:
total_orders = df_clean["Invoice"].nunique()

total_orders

48369

In [73]:
total_customers = df_clean["Customer ID"].nunique()

total_customers

5939

In [74]:
total_products = df_clean["StockCode"].nunique()

total_products

4932

In [75]:
average_order_value = (
    total_revenue / total_orders
)

average_order_value

np.float64(393.10735880419276)

In [76]:
metrics = {
    "Net Revenue": total_revenue,
    "Gross Revenue": gross_revenue,
    "Returned Amount": returned_amount,
    "Orders": total_orders,
    "Customers": total_customers,
    "Products": total_products,
    "Average Order Value": average_order_value
}

metrics

{'Net Revenue': np.float64(19014209.838),
 'Gross Revenue': np.float64(20476260.448),
 'Returned Amount': np.float64(-1462050.61),
 'Orders': 48369,
 'Customers': 5939,
 'Products': 4932,
 'Average Order Value': np.float64(393.10735880419276)}

## Analyze sales over time

Revenue is first aggregated by year. Monthly revenue is then calculated twice: once using month names for a readable display and once using the numeric month so chronological sorting is reliable. The final sort orders each year-month combination correctly.


In [77]:
yearly_sales = (
    df_clean.groupby("Year")["Revenue"]
    .sum()
)

yearly_sales

Year
2009     796648.500
2010    9216135.874
2011    9001425.464
Name: Revenue, dtype: float64

In [78]:
monthly_sales = (
    df_clean.groupby(["Year", "Month_Name"])["Revenue"]
    .sum()
)

monthly_sales

Year  Month_Name
2009  December       796648.500
2010  April          641521.052
      August         654774.390
      December       746723.610
      February       531265.366
      January        622479.502
      July           617365.480
      June           677073.870
      March          763247.241
      May            613270.720
      November      1416697.202
      October       1080611.480
      September      851105.961
2011  April          492367.841
      August         703510.580
      December       432719.060
      February       497026.410
      January        558448.560
      July           680156.991
      June           689977.230
      March          682013.980
      May            722094.100
      November      1456145.800
      October       1069368.230
      September     1017596.682
Name: Revenue, dtype: float64

In [79]:
monthly_sales = (
    df_clean.groupby(["Year", "Month"])["Revenue"]
    .sum()
    .reset_index()
)

monthly_sales

,Year,Month,Revenue
0,2009,12,796648.500
1,2010,1,622479.502
2,2010,2,531265.366
3,2010,3,763247.241
4,2010,4,641521.052
5,2010,5,613270.720
6,2010,6,677073.870
7,2010,7,617365.480
8,2010,8,654774.390
9,2010,9,851105.961


In [80]:
monthly_sales.sort_values(
    ["Year", "Month"]
)

,Year,Month,Revenue
0,2009,12,796648.500
1,2010,1,622479.502
2,2010,2,531265.366
3,2010,3,763247.241
4,2010,4,641521.052
5,2010,5,613270.720
6,2010,6,677073.870
7,2010,7,617365.480
8,2010,8,654774.390
9,2010,9,851105.961


## Compare sales by country

This section sums net revenue by country, ranks markets from largest to smallest, calculates the United Kingdom's share of total revenue, and lists the top countries after excluding the United Kingdom.


In [81]:
country_sales = (
    df_clean.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

country_sales.head(10)

Country
United Kingdom    1.614474e+07
EIRE              6.099538e+05
Netherlands       5.483307e+05
Germany           4.119592e+05
France            3.217334e+05
Australia         1.664444e+05
Switzerland       9.940306e+04
Spain             9.101344e+04
Sweden            8.777552e+04
Denmark           6.445959e+04
Name: Revenue, dtype: float64

In [82]:
uk_share = (
    country_sales["United Kingdom"] /
    country_sales.sum()
) * 100

uk_share

np.float64(84.90879513033805)

In [83]:
country_sales.drop("United Kingdom").head(10)

Country
EIRE           609953.780
Netherlands    548330.700
Germany        411959.161
France         321733.390
Australia      166444.420
Switzerland     99403.060
Spain           91013.440
Sweden          87775.520
Denmark         64459.590
Belgium         63208.890
Name: Revenue, dtype: float64

## Identify top-performing products

Product revenue is initially ranked by description. The following check identifies postage-related descriptions, then creates `df_products` without any description containing “POSTAGE”. The final ranking shows product revenue without delivery charges; because it uses net revenue, returns and cancellations still reduce the relevant product totals.


In [84]:
product_sales = (
    df_clean.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

product_sales.head(10)

Description
REGENCY CAKESTAND 3 TIER               314045.02
DOTCOM POSTAGE                         309844.10
WHITE HANGING HEART T-LIGHT HOLDER     251603.12
PARTY BUNTING                          147079.73
JUMBO BAG RED RETROSPOT                145946.05
ASSORTED COLOUR BIRD ORNAMENT          128550.42
PAPER CHAIN KIT 50'S CHRISTMAS         116298.59
POSTAGE                                110430.41
CHILLI LIGHTS                           79905.13
ROTATING SILVER ANGELS T-LIGHT HLDR     70969.95
Name: Revenue, dtype: float64

In [85]:
df_clean[
    df_clean["Description"].str.contains(
        "POSTAGE",
        case=False,
        na=False
    )
]["Description"].value_counts()

Description
POSTAGE           2079
DOTCOM POSTAGE    1418
Name: count, dtype: int64

In [86]:
df_products = df_clean[
    ~df_clean["Description"].str.contains(
        "POSTAGE",
        case=False,
        na=False
    )
].copy()

In [87]:
product_sales = (
    df_products.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

product_sales.head(10)

Description
REGENCY CAKESTAND 3 TIER               314045.02
WHITE HANGING HEART T-LIGHT HOLDER     251603.12
PARTY BUNTING                          147079.73
JUMBO BAG RED RETROSPOT                145946.05
ASSORTED COLOUR BIRD ORNAMENT          128550.42
PAPER CHAIN KIT 50'S CHRISTMAS         116298.59
CHILLI LIGHTS                           79905.13
ROTATING SILVER ANGELS T-LIGHT HLDR     70969.95
JUMBO BAG STRAWBERRY                    68489.89
RABBIT NIGHT LIGHT                      66661.63
Name: Revenue, dtype: float64

## Prepare the RFM customer base

RFM analysis requires a known customer, so rows with missing customer IDs are removed. The dataset dimensions are displayed to show how many customer-linked transaction lines remain.


In [88]:
df_customers = df_clean[
    df_clean["Customer ID"].notna()
].copy()

In [89]:
df_customers.shape

(797815, 16)

### Calculate Recency, Frequency, and Monetary value

The reference date is set to one day after the latest customer transaction. For each customer, the code calculates:

- **Recency:** days since the customer's most recent transaction.
- **Frequency:** number of distinct invoice identifiers.
- **Monetary:** total net revenue across the customer's transaction lines.

The aggregate column names are then replaced with these standard RFM labels, and the following views inspect typical rows and the highest-spending customers.


In [90]:
analysis_date = df_customers["InvoiceDate"].max() + pd.Timedelta(days=1)

analysis_date

Timestamp('2011-12-10 12:50:00')

In [91]:
rfm = df_customers.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (
        analysis_date - x.max()
    ).days,
    "Invoice": "nunique",
    "Revenue": "sum"
})

In [92]:
rfm.columns = [
    "Recency",
    "Frequency",
    "Monetary"
]

In [93]:
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326,17,-51.74
12347.0,2,8,4921.53
12348.0,75,5,2019.40
12349.0,19,5,4404.54
12350.0,310,1,334.40


In [94]:
rfm.sort_values(
    "Monetary",
    ascending=False
).head(10)

,Recency,Frequency,Monetary
Customer ID,,,
18102.0,1,153,570380.61
14646.0,2,163,523342.07
14156.0,10,202,296063.44
14911.0,1,510,265757.91
17450.0,8,61,231390.55
13694.0,4,164,190020.84
17511.0,3,85,168491.62
12415.0,24,33,143269.29
16684.0,4,65,141502.25


In [95]:
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326,17,-51.74
12347.0,2,8,4921.53
12348.0,75,5,2019.40
12349.0,19,5,4404.54
12350.0,310,1,334.40


> **Interpretation note:** This cell creates a sales-only customer subset by excluding cancellations, but the RFM table above is not recalculated from it. As written, the RFM frequency and monetary values include cancellation transactions; monetary values are net of returns. The following ranking is therefore unchanged and serves as a check of that existing RFM table.


In [96]:
df_customers_sales = df_customers[
    ~df_customers["IsCancellation"]
].copy()

In [97]:
rfm.sort_values(
    "Monetary",
    ascending=False
).head(10)

,Recency,Frequency,Monetary
Customer ID,,,
18102.0,1,153,570380.61
14646.0,2,163,523342.07
14156.0,10,202,296063.44
14911.0,1,510,265757.91
17450.0,8,61,231390.55
13694.0,4,164,190020.84
17511.0,3,85,168491.62
12415.0,24,33,143269.29
16684.0,4,65,141502.25


### Score each RFM dimension

Customers are divided into five ranked groups (quintiles) for recency, frequency, and monetary value. Recency receives a higher score for fewer days since the last purchase, while frequency and monetary value receive higher scores for larger values. Ranking with `method="first"` breaks ties so the quintile calculation can assign every customer. The three scores are concatenated into a compact RFM score such as `555`.


In [99]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5,4,3,2,1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

In [100]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    +
    rfm["F_Score"].astype(str)
    +
    rfm["M_Score"].astype(str)
)

In [101]:
rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
Customer ID,,,,,,,
12346.0,326,17,-51.74,2,5,1,251
12347.0,2,8,4921.53,5,4,5,545
12348.0,75,5,2019.40,3,3,4,334
12349.0,19,5,4404.54,4,3,5,435
12350.0,310,1,334.40,2,1,2,212


### Assign business-friendly customer segments

The function maps score combinations to segments: high scorers on all three measures are **Champions**; customers with strong recency and frequency are **Loyal Customers**; highly recent customers are **Recent Customers**; less-recent but frequent buyers are **At Risk**; and all remaining customers are labelled **Lost Customers**. The next cells apply the rule to each customer and count the resulting segments.


In [102]:
def segment_customer(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"
    
    elif row["R_Score"] >= 3 and row["F_Score"] >= 3:
        return "Loyal Customers"
    
    elif row["R_Score"] >= 4:
        return "Recent Customers"
    
    elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"
    
    else:
        return "Lost Customers"

In [103]:
rfm["Segment"] = rfm.apply(
    segment_customer,
    axis=1
)

In [104]:
rfm["Segment"].value_counts()

Segment
Lost Customers      1919
Loyal Customers     1421
Champions           1301
At Risk              841
Recent Customers     457
Name: count, dtype: int64

## Build the reporting tables and export the workbook

Three output tables are prepared: cleaned transaction lines (`sales`), customer RFM scores and segments (`customers`), and product-level quantity/revenue totals excluding postage (`products`). The final cell writes them to separate **Sales**, **Customers**, and **Products** sheets in `Retail_Analytics_Model.xlsx`.


In [105]:
sales = df_clean.copy()

In [106]:
customers = rfm.reset_index()

In [107]:
products = (
    df_products
    .groupby(["StockCode","Description"])
    .agg(
        Total_Quantity=("Quantity","sum"),
        Total_Revenue=("Revenue","sum")
    )
    .reset_index()
)

In [112]:
with pd.ExcelWriter("Retail_Analytics_Model.xlsx") as writer:
    sales.to_excel(
        writer,
        sheet_name="Sales",
        index=False
    )

    customers.to_excel(
        writer,
        sheet_name="Customers",
        index=False
    )

    products.to_excel(
        writer,
        sheet_name="Products",
        index=False
    )